# 01 — Data Collection
Pulls posts from r/SkincareAddiction via Reddit API (PRAW) and loads them into PostgreSQL.

In [ ]:
import os
import praw
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

In [ ]:
# --- Reddit API connection ---
reddit = praw.Reddit(
    client_id=os.getenv('REDDIT_CLIENT_ID'),
    client_secret=os.getenv('REDDIT_CLIENT_SECRET'),
    user_agent=os.getenv('REDDIT_USER_AGENT'),
)
print('Read-only mode:', reddit.read_only)

In [ ]:
# --- PostgreSQL connection ---
DB_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

In [ ]:
# --- Skin concern keyword mapping ---
CONCERN_KEYWORDS = {
    'acne':             ['acne', 'breakout', 'pimple', 'comedone', 'whitehead', 'blackhead', 'cystic'],
    'aging':            ['wrinkle', 'fine line', 'anti-aging', 'collagen', 'retinol', 'retinoid'],
    'sensitivity':      ['sensitive', 'redness', 'irritation', 'rosacea', 'eczema', 'reactive'],
    'hyperpigmentation':['dark spot', 'hyperpigmentation', 'melasma', 'uneven tone', 'brightening'],
}

def classify_concern(text: str) -> str:
    text_lower = text.lower()
    for concern, keywords in CONCERN_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return concern
    return 'other'

In [ ]:
# --- Pull posts ---
SUBREDDIT = 'SkincareAddiction'
LIMIT = 1000   # start small, scale up

records = []
subreddit = reddit.subreddit(SUBREDDIT)

for post in tqdm(subreddit.new(limit=LIMIT), total=LIMIT, desc='Collecting posts'):
    full_text = (post.title or '') + ' ' + (post.selftext or '')
    records.append({
        'post_id':      post.id,
        'subreddit':    SUBREDDIT,
        'title':        post.title,
        'body':         post.selftext,
        'score':        post.score,
        'num_comments': post.num_comments,
        'created_at':   datetime.utcfromtimestamp(post.created_utc),
        'skin_concern': classify_concern(full_text),
    })

df = pd.DataFrame(records)
print(f'Collected {len(df)} posts')
print(df['skin_concern'].value_counts())

In [ ]:
# --- Save raw data ---
df.to_json('../data/raw/posts_raw.json', orient='records', indent=2)

# --- Load into PostgreSQL ---
df.to_sql('posts', engine, if_exists='append', index=False)
print('Loaded to PostgreSQL ✓')